# Ingest `drivers.json` file

1. Read the file using spark dataframe reader API
2. Define and enforce schema (preserve the nested structure)
3. Add metadata columns
    - Source File
    - Ingestion Timestamp
4. Write to Bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

## Define source_file and table_name

In [0]:
source_file = f'{landing_folder_path}/drivers.json'
table_name = f'{catalog_name}.{bronze_schema}.drivers'

## Step 1 - Read JSON file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType
name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

drivers_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])


In [0]:
# Read data from the constructors file
drivers_df = (spark.read
                   .format('json')
                   .schema(drivers_schema)
                   .option('mode', 'FAILFAST')
                   .load(source_file)
                   )

In [0]:
display(drivers_df)

## Step 2 - Add Metadata Columns

In [0]:
drivers_final_df = add_ingetion_metadata(drivers_df)

In [0]:
display(drivers_final_df)

## Step 3 - Write to bronze delta table

In [0]:
(
    drivers_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))